# SABR Paper Reproduction Walkthrough

This notebook organizes the current project into a single step-by-step workflow so the paper replication can be run one cell at a time.

Paper:
- Choi, Hu, Kwok, *Efficient and accurate simulation of the stochastic-alpha-beta-rho model*

Project scope:
- core SABR Monte Carlo simulator
- sanity checks for limiting cases and martingale behavior
- paper-style tables and figure datasets
- quick and paper-scale validation hooks


## Contents

1. [Environment and Imports](#1.-Environment-and-Imports)
2. [Project API Overview](#2.-Project-API-Overview)
3. [Core Mathematical Formulas](#3.-Core-Mathematical-Formulas)
4. [Table 3 Cases](#4.-Table-3-Cases)
5. [Sanity Checks](#5.-Sanity-Checks)
6. [Paper Table 1](#6.-Paper-Table-1)
7. [Paper Table 2](#7.-Paper-Table-2)
8. [Paper Figure 1 Dataset](#8.-Paper-Figure-1-Dataset)
9. [Paper Table 4](#9.-Paper-Table-4)
10. [Paper Table 5](#10.-Paper-Table-5)
11. [Paper Table 6](#11.-Paper-Table-6)
12. [Paper Table 7 / Figure 2](#12.-Paper-Table-7-/-Figure-2)
13. [Paper Figure 3](#13.-Paper-Figure-3)
14. [Validation Summary](#14.-Validation-Summary)
15. [Asian Option Extension](#15.-Asian-Option-Extension)
16. [Optional CSV Export Helpers](#16.-Optional-CSV-Export-Helpers)


## 1. Environment and Imports

Run this first. It checks the working directory, imports the project module, and shows package versions when available.


In [ ]:
from pathlib import Path
import sys
import platform
import importlib
import urllib.request
import subprocess

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path('/content')
    project_file = project_root / 'sabr_replicate.py'
    raw_url = 'https://raw.githubusercontent.com/numerical-project-team/numerical-project/pyfeng-refactor-moments/sabr_replicate.py'
    urllib.request.urlretrieve(raw_url, project_file)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'pyfeng', 'statsmodels', 'matplotlib', 'mafn-finite-difference',
    ], check=True)
else:
    NOTEBOOK_DIR = Path.cwd()
    project_root = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop('sabr_replicate', None)

import numpy as np
import pandas as pd

PROJECT_ROOT = project_root

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Project root:', PROJECT_ROOT)
print('sabr_replicate.py exists:', (PROJECT_ROOT / 'sabr_replicate.py').exists())

for pkg in ['numpy', 'pandas', 'scipy', 'pyfeng', 'statsmodels', 'matplotlib']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}:', getattr(mod, '__version__', 'version unavailable'))


In [ ]:
if not (PROJECT_ROOT / 'sabr_replicate.py').exists():
    raise FileNotFoundError(
        'sabr_replicate.py was not found. Run the Environment and Imports cell first.'
    )

from sabr_replicate import (
    FDMConfig,
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    conditional_integrated_variance_stats,
    european_call_price,
    fdm_benchmark_prices,
    figure1_moment_comparison,
    figure2_runtime_tradeoff,
    finite_difference_call_price,
    martingale_test,
    run_figure3_experiment,
    run_full_validation,
    run_table1_experiment,
    run_table2_experiment,
    run_table4_experiment,
    run_table5_experiment,
    run_table6_experiment,
    run_table7_experiment,
    sample_conditional_integrated_variance,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)
import pyfeng as pf

print('Project imports loaded from repository checkout.')


## 2. Project API Overview

This section is only for orientation. It lists the main functions used later in the notebook.


## 3. Core Mathematical Formulas

This section collects the formulas that the implementation is reproducing. In the code, the same objects appear as `SABRParams`, `MonteCarloConfig`, `sample_conditional_integrated_variance`, `sample_cev_exact`, and `simulate_terminal_forward`.

### 3.1 SABR model

The SABR dynamics are

$$
dF_t = \sigma_t F_t^\beta\, dW_t,
\qquad
\frac{d\sigma_t}{\sigma_t} = \nu\, dZ_t,
\qquad
dW_t\,dZ_t = \rho\,dt .
$$

The exact volatility step over one interval of length $h$ is

$$
\sigma_{t+h}
=
\sigma_t
\exp\left(\nu\sqrt{h}X_\sigma - \frac{1}{2}\nu^2 h\right),
\qquad X_\sigma \sim N(0,1).
$$

We use

$$
\hat\nu = \nu\sqrt{h},
\qquad
\beta^* = 1-\beta,
\qquad
\rho^* = \sqrt{1-\rho^2}.
$$

### 3.2 Algorithm 1: conditional average variance

After sampling $\sigma_{t+h}$, the first hard quantity is the normalized conditional average variance

$$
I_t^h
=
\frac{1}{\sigma_t^2 h}
\int_t^{t+h} \sigma_s^2\,ds
\;\Bigg|\;\sigma_{t+h}.
$$

Let

$$
\mu = \mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right],
\qquad
v =
\frac{
\sqrt{\mathrm{Var}\left(I_t^h \mid \sigma_{t+h}\right)}
}{
\mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right]
}.
$$

With fixed shift

$$
\lambda = \frac{5}{6},
$$

the lognormal shape parameter is

$$
a
=
\sqrt{\log\left(1 + \frac{v^2}{\lambda^2}\right)}
=
\sqrt{\log\left(1 + \frac{36}{25}v^2\right)}.
$$

Algorithm 1 samples

$$
I_t^h
\approx
\mu
\left[
\frac{1}{6}
+
\frac{5}{6}
\exp\left(aX - \frac{1}{2}a^2\right)
\right],
\qquad X\sim N(0,1).
$$

In the implementation, `conditional_integrated_variance_stats` supplies PyFeng-backed moment statistics and `sample_conditional_integrated_variance` performs this shifted-lognormal sampling.

### 3.3 Algorithm 2: martingale-preserving CEV approximation

For $0<\beta<1$, conditional on $\sigma_{t+h}$ and $I_t^h$,

$$
F_{t+h}\mid \sigma_{t+h}, I_t^h
\approx
\mathrm{CEV}_\beta\left(\bar F_t^h, (\rho^*)^2\sigma_t^2 h I_t^h\right).
$$

The conditional mean is

$$
\bar F_t^h
\approx
F_t
\exp\left(
\frac{\rho(\sigma_{t+h}-\sigma_t)}{\nu F_t^{\beta^*}}
-
\frac{\rho^2\sigma_t^2 h I_t^h}{2F_t^{2\beta^*}}
\right).
$$

This construction is meant to preserve the martingale condition:

$$
\mathbb{E}[F_{t+h}\mid \sigma_{t+h}, I_t^h] = \bar F_t^h,
\qquad
\mathbb{E}[F_{t+h}] = F_t.
$$

For the lognormal special case $\beta=1$,

$$
F_{t+h}
=
\bar F_t^h
\exp\left(
\rho^*\sigma_t\sqrt{hI_t^h}X
-
\frac{1}{2}(\rho^*)^2\sigma_t^2 hI_t^h
\right),
\qquad X\sim N(0,1).
$$

### 3.4 Algorithm 3: exact CEV sampling

For
$$
F_T \sim \mathrm{CEV}_\beta(F_0, \sigma_0^2T),
$$

define
$$
\alpha = \frac{1}{2\beta^*},
\qquad
z_0 = \frac{F_0^{2\beta^*}}{(\beta^*)^2\sigma_0^2T}.
$$

Sample
$$
X \sim \Gamma(\alpha,1),
$$
where $\Gamma(\alpha,1)$ denotes a Gamma distribution with shape $\alpha$ and unit scale.

If $X \ge z_0/2$, set $F_T = 0$ (absorbing boundary). Otherwise sample
$$
N \sim \mathrm{Poisson}(z_0/2 - X),
\qquad
z_T \sim 2\Gamma(N+1,1),
$$

and return
$$
F_T = \left((\beta^*)^2\sigma_0^2Tz_T\right)^{1/(2\beta^*)}.
$$

This provides an exact sampling scheme for the CEV distribution via a Poisson–Gamma mixture representation.

Inside SABR, this sampler is applied with
$$
F_0 \leftarrow \bar F_t^h,
\qquad
\sigma_0^2T \leftarrow (\rho^*)^2\sigma_t^2hI_t^h.
$$

### 3.5 Algorithm 4: full simulation over a time grid

For each time step:

1. Sample $\sigma_{t+h}$ exactly.
2. Sample $I_t^h$ using Algorithm 1.
3. Compute $\bar F_t^h$ using Algorithm 2.
4. Sample $F_{t+h}$ using Algorithm 3.
5. Repeat until maturity $T$.

The Monte Carlo European call estimator is

$$
\widehat C(K)
=
\frac{1}{N}\sum_{i=1}^N \max(F_T^{(i)}-K,0).
$$

The reported errors are

$$
\text{bias} = \widehat C - C_{\text{benchmark}},
\qquad
\text{relative error} = \frac{\widehat C-C_{\text{benchmark}}}{C_{\text{benchmark}}},
$$

$$
\text{RMS error} = \sqrt{\text{bias}^2 + \text{stdev}^2}.
$$

The martingale check verifies numerically that

$$
\mathbb{E}[F_T] = F_0.
$$


In [ ]:
api_overview = pd.DataFrame(
    [
        ('simulate_terminal_forward', 'Main SABR Monte Carlo terminal simulator using the paper scheme'),
        ('simulate_terminal_forward_islah', 'Appendix B / Islah-style comparison branch'),
        ('sample_conditional_integrated_variance', 'Algorithm 1 style conditional average-variance sampling'),
        ('conditional_integrated_variance_stats', 'PyFeng-backed conditional average-variance statistics'),
        ('run_table1_experiment', 'Paper Table 1 dataset'),
        ('run_table2_experiment', 'Paper Table 2 dataset'),
        ('run_table4_experiment', 'Paper Table 4 dataset'),
        ('run_table5_experiment', 'Paper Table 5 dataset'),
        ('run_table6_experiment', 'Paper Table 6 dataset'),
        ('run_table7_experiment', 'Paper Table 7 / Figure 2 dataset'),
        ('run_figure3_experiment', 'Paper Figure 3 dataset'),
        ('run_full_validation', 'Repository validation harness'),
    ],
    columns=['function', 'purpose'],
)
api_overview


## 4. Table 3 Cases

These are the paper parameter presets used throughout later sections.


In [ ]:
case_df = pd.DataFrame(case_table_3()).T
case_df.index.name = 'case'
case_df


## 5. Sanity Checks

These cells verify model-level behavior that should hold independently of the paper tables.


### 5.1 `nu = 0` should reduce SABR to CEV


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
cev_prices = pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'cev_price': cev_prices,
    'error': mc_prices - cev_prices,
})


### 5.2 `beta = 1, nu = 0` should reduce SABR to Black-Scholes / lognormal


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
bsm_prices = pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'bsm_price': bsm_prices,
    'error': mc_prices - bsm_prices,
})


### 5.3 Conditional average-variance moment spot check

This is a small diagnostic cell for the `I_t^h` machinery used inside the full simulator.


In [ ]:
sigma_t = np.array([0.2])
sigma_next = np.array([0.24])
nu = 0.4
h = 1.0

mean, cv, skewness, ex_kurtosis = conditional_integrated_variance_stats(sigma_t, sigma_next, nu, h)
var = (cv * mean) ** 2
std = np.sqrt(var)

pd.DataFrame({
    'mean': mean,
    'variance': var,
    'std': std,
    'cv': cv,
    'skewness': skewness,
    'ex_kurtosis': ex_kurtosis,
})


### 5.4 Martingale sanity check


In [ ]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)


### 5.5 `|rho| = 1` Islah edge case stability


In [ ]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append({
        'beta': beta,
        'all_finite': bool(np.isfinite(terminal).all()),
        'all_nonnegative': bool((terminal >= 0.0).all()),
        'mean_terminal': float(np.mean(terminal)),
        'min_terminal': float(np.min(terminal)),
    })

pd.DataFrame(rows)


## 6. Paper Table 1

By default, this uses the paper table benchmarks embedded in the repo. To compare against the repo PDE benchmark instead, run the later FDM cells or pass benchmark providers in standalone scripts.


In [ ]:
table1_df = run_table1_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table1_df


## 7. Paper Table 2


In [ ]:
table2_df = run_table2_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table2_df


## 8. Paper Figure 1 Dataset

This generates the skewness and ex-kurtosis comparison data used for the figure. Plotting is optional; the dataset itself is enough for validation.


In [ ]:
figure1_df = figure1_moment_comparison(hat_nu=0.4)
figure1_df.head(12)


In [ ]:
ax = figure1_df.plot(x='z_hat', y=['exact_skewness', 'ln_skewness', 'sln_fixed_skewness', 'sln_exact_skewness'], figsize=(10, 4), title='Figure 1 style skewness comparison')
ax.set_ylabel('skewness')


## 9. Paper Table 4

Case I. This includes the simulated rows and the available analytic comparison rows.


In [ ]:
table4_df = run_table4_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table4_df


## 10. Paper Table 5

Case II.


In [ ]:
table5_df = run_table5_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table5_df


## 11. Paper Table 6

Case III. This is the runtime / bias comparison against paper-reference baseline rows.


In [ ]:
table6_df = run_table6_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table6_df


## 12. Paper Table 7 / Figure 2

This section can be slow. It reproduces the convergence and runtime trade-off dataset.


In [ ]:
table7_df = run_table7_experiment(
    n_paths_base=160_000,
    n_repeats=10,
    seed0=12345,
    benchmark_source='fdm',
)
table7_df


In [ ]:
figure2_df = figure2_runtime_tradeoff(
    n_paths_base=160_000,
    n_repeats=10,
    seed0=12345,
    benchmark_source='fdm',
)
figure2_df


### Figure 2 Plot

This turns the Table 7 / Figure 2 dataset into the paper-style runtime versus RMS error view.

In [ ]:
import matplotlib.pyplot as plt

fig2_plot_df = figure2_df.dropna(subset=['runtime_sec_mean', 'rms_error_x1e3']).copy()
fig2_plot_df['series'] = fig2_plot_df['source'].map({
    'simulated': 'Our run',
    'paper_reference': 'Paper reference',
}).fillna(fig2_plot_df['source'])

fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in fig2_plot_df.groupby('series'):
    grp = grp.sort_values('runtime_sec_mean')
    ax.plot(grp['runtime_sec_mean'], grp['rms_error_x1e3'], marker='o', label=label)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Simulation time / runtime per repeat (sec)')
ax.set_ylabel('RMS error x 1e3')
ax.set_title('Figure 2 style: runtime vs RMS error')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.show()

## 13. Paper Figure 3

This compares the paper scheme against the Islah approximation across maturities.


In [ ]:
figure3_df = run_figure3_experiment(
    n_paths=100_000,
    n_repeats=2,
    seed0=12345,
    benchmark_source='mc',
)
figure3_df.head(20)


In [ ]:
pivot_forward = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='forward_error')
pivot_option = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='option_error')

display(pivot_forward)
display(pivot_option)


### Figure 3 Plots

These plots recreate the paper-style Figure 3 comparison: panel (a) is forward-price error, and panel (b) is ATM option-price error.

In [ ]:
import matplotlib.pyplot as plt

ax = pivot_forward.plot(figsize=(8, 4), marker='o', title='Figure 3(a): forward-price error')
ax.axhline(0.0, color='black', linewidth=0.8)
ax.set_xlabel('Maturity')
ax.set_ylabel('E[F_T] - F0')
ax.grid(True, alpha=0.3)
plt.show()

ax = pivot_option.plot(figsize=(8, 4), marker='o', title='Figure 3(b): ATM option-price error')
ax.axhline(0.0, color='black', linewidth=0.8)
ax.set_xlabel('Maturity')
ax.set_ylabel('Option price error')
ax.grid(True, alpha=0.3)
plt.show()

## 14. Validation Summary

Start with quick mode. The paper-scale run is much slower.


In [ ]:
validation_quick = run_full_validation(quick_mode=True)
pd.Series({
    'table1_status': validation_quick['table1_status'],
    'table2_status': validation_quick['table2_status'],
    'overall_conclusion': validation_quick['overall_conclusion'],
    'replication_conclusion': validation_quick['replication_conclusion'],
})


In [ ]:
# Uncomment this only when you want the full validation sweep.
# validation_full = run_full_validation(quick_mode=False)
# pd.Series({
#     'table1_status': validation_full['table1_status'],
#     'table2_status': validation_full['table2_status'],
#     'overall_conclusion': validation_full['overall_conclusion'],
#     'replication_conclusion': validation_full['replication_conclusion'],
# })


## 15. Asian Option Extension

This section merges the latest Asian option extension into the live reproduction notebook. It keeps the paper's SABR transition machinery, but applies it to a discretely monitored arithmetic Asian call and adds numerical diagnostics for consistency, martingale behavior, monitoring-frequency convergence, and variance reduction.


In [ ]:
import math
import time
import matplotlib.pyplot as plt

from sabr_replicate import (
    EPS,
    PDF_FLOOR,
    _correlated_drift_term,
    sample_cev_exact,
    sample_sigma_next,
)

pd.options.display.float_format = '{:.6f}'.format
plt.rcParams.update({
    'figure.figsize': (8, 4.6),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


### 15.1 Contract and Estimator Helpers

The contract convention is intentionally explicit:

- arithmetic average over future monitoring dates only by default;
- no discounting inside the payoff;
- the expected arithmetic average should be close to $F_0$ because the simulated forward is designed to preserve the martingale property.

That last point gives a useful control variate: since $E[A_T] = F_0$, the variable $A_T - F_0$ can reduce the Monte Carlo standard error of the Asian payoff.


In [ ]:
class AsianContract:
    def __init__(self, maturity: float, strike: float, n_monitoring: int, include_initial: bool = False):
        self.maturity = maturity
        self.strike = strike
        self.n_monitoring = n_monitoring
        self.include_initial = include_initial
        self.step = maturity / n_monitoring
        self.average_denominator = n_monitoring + int(include_initial)


class AsianPathSamples:
    def __init__(
        self,
        terminal: np.ndarray,
        arithmetic_average: np.ndarray,
        runtime_sec: float,
        absorbed_fraction: float,
    ):
        self.terminal = terminal
        self.arithmetic_average = arithmetic_average
        self.runtime_sec = runtime_sec
        self.absorbed_fraction = absorbed_fraction


def mean_estimate(samples: np.ndarray, expected: float | None = None) -> dict[str, float]:
    samples = np.asarray(samples, dtype=float)
    mean = float(np.mean(samples))
    stderr = float(np.std(samples, ddof=1) / math.sqrt(samples.size))
    out = {
        'mean': mean,
        'stderr': stderr,
        'ci95_low': mean - 1.96 * stderr,
        'ci95_high': mean + 1.96 * stderr,
    }
    if expected is not None:
        out['expected'] = float(expected)
        out['z_score'] = float((mean - expected) / stderr) if stderr > 0 else np.nan
    return out


def call_payoff(underlying: np.ndarray, strike: float) -> np.ndarray:
    return np.maximum(np.asarray(underlying, dtype=float) - strike, 0.0)


def mc_price_from_payoff(payoff: np.ndarray) -> dict[str, float]:
    payoff = np.asarray(payoff, dtype=float)
    price = float(np.mean(payoff))
    stderr = float(np.std(payoff, ddof=1) / math.sqrt(payoff.size))
    return {
        'price': price,
        'stderr': stderr,
        'ci95_low': price - 1.96 * stderr,
        'ci95_high': price + 1.96 * stderr,
    }


def asian_price_raw(average: np.ndarray, strike: float) -> dict[str, float]:
    return mc_price_from_payoff(call_payoff(average, strike))


def asian_price_with_average_control(
    average: np.ndarray,
    strike: float,
    expected_average: float,
) -> dict[str, float]:
    payoff = call_payoff(average, strike)
    control = np.asarray(average, dtype=float) - expected_average
    control_var = float(np.var(control, ddof=1))
    if control_var <= 0.0:
        base = mc_price_from_payoff(payoff)
        base.update({'beta_cv': 0.0, 'stderr_reduction': 1.0})
        return base

    beta_cv = float(np.cov(payoff, control, ddof=1)[0, 1] / control_var)
    adjusted = payoff - beta_cv * control
    cv = mc_price_from_payoff(adjusted)
    raw = mc_price_from_payoff(payoff)
    cv.update({
        'beta_cv': beta_cv,
        'raw_stderr': raw['stderr'],
        'stderr_reduction': raw['stderr'] / cv['stderr'] if cv['stderr'] > 0 else np.inf,
    })
    return cv


def european_price_estimate(terminal: np.ndarray, strike: float) -> dict[str, float]:
    return mc_price_from_payoff(call_payoff(terminal, strike))


### 15.2 Path Simulator Using the Paper Transition

The original project function `simulate_terminal_forward` returns only $F_T$. For an Asian payoff, we need the running arithmetic average. The function below repeats the paper one-step transition and records the average along the monitoring grid.

A short single-step check follows the implementation: when `n_monitoring = 1`, the terminal samples should match the project terminal simulator exactly for the same seed.


In [ ]:
def simulate_terminal_and_arithmetic_average_paper_scheme(
    params: SABRParams,
    contract: AsianContract,
    n_paths: int,
    seed: int = 12345,
) -> AsianPathSamples:
    if not (0.0 < params.beta < 1.0):
        raise ValueError('This notebook focuses on the main paper case 0 < beta < 1.')
    if contract.maturity <= 0.0:
        raise ValueError('maturity must be positive')
    if contract.n_monitoring <= 0:
        raise ValueError('n_monitoring must be positive')
    if n_paths <= 1:
        raise ValueError('n_paths must be larger than 1')

    rng = np.random.default_rng(seed)
    h = contract.step
    beta_star = params.beta_star
    rho_star_sq = max(0.0, 1.0 - params.rho * params.rho)

    f = np.full(n_paths, params.f0, dtype=float)
    sigma = np.full(n_paths, params.sigma0, dtype=float)
    running_sum = np.full(n_paths, params.f0 if contract.include_initial else 0.0, dtype=float)

    start = time.perf_counter()

    for _ in range(contract.n_monitoring):
        absorbed = f <= 0.0
        sigma_next = sample_sigma_next(sigma, params.nu, h, rng)

        if np.all(absorbed):
            sigma = sigma_next
            running_sum += f
            continue

        f_alive = f[~absorbed]
        sigma_alive = sigma[~absorbed]
        sigma_next_alive = sigma_next[~absorbed]

        if abs(params.nu) < EPS:
            variance_scale = sigma_alive * sigma_alive * h
            f_next_alive = sample_cev_exact(f_alive, variance_scale, params.beta, rng)
        else:
            ih = sample_conditional_integrated_variance(
                sigma_alive,
                sigma_next_alive,
                params.nu,
                h,
                rng,
            )
            f_pow = np.maximum(f_alive, PDF_FLOOR) ** beta_star
            drift = _correlated_drift_term(
                sigma_alive,
                sigma_next_alive,
                params.nu,
                f_pow,
                params.rho,
            )
            variance_scale = rho_star_sq * sigma_alive * sigma_alive * h * ih
            f_bar = f_alive * np.exp(
                drift
                - 0.5 * params.rho * params.rho * sigma_alive * sigma_alive * h * ih
                / np.maximum(f_pow * f_pow, PDF_FLOOR)
            )
            f_bar = np.maximum(f_bar, 0.0)
            if rho_star_sq < EPS:
                f_next_alive = f_bar
            else:
                f_next_alive = sample_cev_exact(f_bar, variance_scale, params.beta, rng)

        f[~absorbed] = f_next_alive
        sigma = sigma_next
        running_sum += f

    runtime_sec = time.perf_counter() - start
    average = running_sum / contract.average_denominator
    absorbed_fraction = float(np.mean(f <= 0.0))
    return AsianPathSamples(
        terminal=f,
        arithmetic_average=average,
        runtime_sec=runtime_sec,
        absorbed_fraction=absorbed_fraction,
    )


In [ ]:
consistency_params = SABRParams(f0=1.0, sigma0=0.25, nu=0.5, rho=-0.5, beta=0.6)
consistency_contract = AsianContract(maturity=2.0, strike=1.0, n_monitoring=1)
consistency_paths = 20_000
consistency_seed = 20260503

asian_one_step = simulate_terminal_and_arithmetic_average_paper_scheme(
    consistency_params,
    consistency_contract,
    n_paths=consistency_paths,
    seed=consistency_seed,
)
terminal_only = simulate_terminal_forward(
    consistency_params,
    MonteCarloConfig(
        maturity=consistency_contract.maturity,
        step=consistency_contract.maturity,
        n_paths=consistency_paths,
        seed=consistency_seed,
    ),
)

single_step_check = pd.DataFrame([
    {
        'check': 'n_monitoring=1 matches simulate_terminal_forward',
        'max_abs_terminal_diff': float(np.max(np.abs(asian_one_step.terminal - terminal_only))),
        'price_diff_at_K': float(
            european_call_price(asian_one_step.terminal, consistency_contract.strike)
            - european_call_price(terminal_only, consistency_contract.strike)
        ),
        'passed': bool(np.allclose(asian_one_step.terminal, terminal_only, rtol=0.0, atol=1e-13)),
    }
])
single_step_check


### 15.3 Baseline Asian Pricing Run

The baseline case uses a two-year maturity, negative spot-vol correlation, and a non-lognormal elasticity. This is a useful moderately stressed case because the option is path dependent, the average is monitored monthly, and the correlation term in the CEV approximation matters.


In [ ]:
params = SABRParams(
    f0=1.0,
    sigma0=0.25,
    nu=0.5,
    rho=-0.5,
    beta=0.6,
)
contract = AsianContract(maturity=2.0, strike=1.0, n_monitoring=24, include_initial=False)
n_paths = 100_000
seed = 777_001

samples = simulate_terminal_and_arithmetic_average_paper_scheme(
    params,
    contract,
    n_paths=n_paths,
    seed=seed,
)

asian_raw = asian_price_raw(samples.arithmetic_average, contract.strike)
asian_cv = asian_price_with_average_control(samples.arithmetic_average, contract.strike, params.f0)
european_same_paths = european_price_estimate(samples.terminal, contract.strike)

baseline_table = pd.DataFrame([
    {
        'payoff': 'European call on F_T',
        'price': european_same_paths['price'],
        'stderr': european_same_paths['stderr'],
        'ci95_low': european_same_paths['ci95_low'],
        'ci95_high': european_same_paths['ci95_high'],
        'control': 'none',
    },
    {
        'payoff': 'Arithmetic Asian call',
        'price': asian_raw['price'],
        'stderr': asian_raw['stderr'],
        'ci95_low': asian_raw['ci95_low'],
        'ci95_high': asian_raw['ci95_high'],
        'control': 'none',
    },
    {
        'payoff': 'Arithmetic Asian call',
        'price': asian_cv['price'],
        'stderr': asian_cv['stderr'],
        'ci95_low': asian_cv['ci95_low'],
        'ci95_high': asian_cv['ci95_high'],
        'control': 'A_T - F0',
    },
])

run_metadata = pd.DataFrame([
    {
        'maturity': contract.maturity,
        'strike': contract.strike,
        'n_monitoring': contract.n_monitoring,
        'step_size': contract.step,
        'n_paths': n_paths,
        'seed': seed,
        'runtime_sec': samples.runtime_sec,
        'absorbed_fraction': samples.absorbed_fraction,
        'cv_beta': asian_cv['beta_cv'],
        'cv_stderr_reduction': asian_cv['stderr_reduction'],
    }
])

display(run_metadata)
baseline_table


### 15.4 Martingale and Average Consistency Diagnostics

The paper's CEV approximation is designed so that each forward step preserves the martingale condition. For an equally weighted arithmetic average of future monitoring dates, this implies $E[A_T] = F_0$ as well. The table below reports z-scores rather than only raw means, so the diagnostic separates visible bias from ordinary Monte Carlo noise.


In [ ]:
terminal_diag = mean_estimate(samples.terminal, expected=params.f0)
average_diag = mean_estimate(samples.arithmetic_average, expected=params.f0)

diagnostic_table = pd.DataFrame([
    {
        'quantity': 'terminal forward F_T',
        **terminal_diag,
    },
    {
        'quantity': 'arithmetic average A_T',
        **average_diag,
    },
])

diagnostic_table


### 15.5 Monitoring-Frequency Convergence

Asian prices depend on the monitoring grid. There is no simple FDM benchmark for the arithmetic Asian payoff under SABR, so this section uses a fine-grid Monte Carlo run as an internal benchmark. The RMS column combines the bias against the fine-grid proxy and the Monte Carlo standard errors of both estimates.

This mirrors the paper's error accounting idea: price error should be interpreted together with sampling uncertainty and runtime, not only as a raw point estimate.


In [ ]:
reference_contract = AsianContract(
    maturity=contract.maturity,
    strike=contract.strike,
    n_monitoring=252,
    include_initial=contract.include_initial,
)
reference_paths = 80_000
reference_seed = 880_000
reference_samples = simulate_terminal_and_arithmetic_average_paper_scheme(
    params,
    reference_contract,
    n_paths=reference_paths,
    seed=reference_seed,
)
reference_cv = asian_price_with_average_control(
    reference_samples.arithmetic_average,
    reference_contract.strike,
    params.f0,
)

monitoring_grid = [12, 24, 52, 126]
grid_paths = 50_000
rows = []
for i, n_monitoring in enumerate(monitoring_grid):
    c = AsianContract(
        maturity=contract.maturity,
        strike=contract.strike,
        n_monitoring=n_monitoring,
        include_initial=contract.include_initial,
    )
    s = simulate_terminal_and_arithmetic_average_paper_scheme(
        params,
        c,
        n_paths=grid_paths,
        seed=881_000 + i,
    )
    raw = asian_price_raw(s.arithmetic_average, c.strike)
    cv = asian_price_with_average_control(s.arithmetic_average, c.strike, params.f0)
    avg_diag = mean_estimate(s.arithmetic_average, expected=params.f0)
    bias = cv['price'] - reference_cv['price']
    rms = math.sqrt(bias * bias + cv['stderr'] ** 2 + reference_cv['stderr'] ** 2)
    rows.append({
        'n_monitoring': n_monitoring,
        'step_size': c.step,
        'price_raw': raw['price'],
        'stderr_raw': raw['stderr'],
        'price_cv': cv['price'],
        'stderr_cv': cv['stderr'],
        'bias_vs_252_cv': bias,
        'rms_vs_252_cv': rms,
        'mean_average_z': avg_diag['z_score'],
        'runtime_sec': s.runtime_sec,
        'stderr_reduction': cv['stderr_reduction'],
    })

monitoring_df = pd.DataFrame(rows)
reference_summary = pd.DataFrame([
    {
        'n_monitoring': reference_contract.n_monitoring,
        'step_size': reference_contract.step,
        'price_cv': reference_cv['price'],
        'stderr_cv': reference_cv['stderr'],
        'n_paths': reference_paths,
        'runtime_sec': reference_samples.runtime_sec,
    }
])

display(reference_summary)
monitoring_df


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    monitoring_df['n_monitoring'],
    monitoring_df['price_cv'],
    yerr=1.96 * monitoring_df['stderr_cv'],
    marker='o',
    capsize=4,
    label='Coarser monitoring grids',
)
ax.axhline(reference_cv['price'], color='black', linestyle='--', linewidth=1.2, label='252-date reference')
ax.fill_between(
    [monitoring_df['n_monitoring'].min(), monitoring_df['n_monitoring'].max()],
    reference_cv['price'] - 1.96 * reference_cv['stderr'],
    reference_cv['price'] + 1.96 * reference_cv['stderr'],
    color='black',
    alpha=0.08,
)
ax.set_xlabel('Number of monitoring dates')
ax.set_ylabel('Asian call price')
ax.set_title('Monitoring-frequency convergence')
ax.legend()
plt.show()


### 15.6 Strike Sweep and Shape Checks

A realistic pricing extension should check more than one strike. The same simulated averages are reused across strikes, which makes monotonicity and convexity checks more meaningful because the payoff is evaluated pathwise on a shared sample.


In [ ]:
strike_grid = np.linspace(0.70, 1.30, 13)
strike_rows = []
for k in strike_grid:
    raw = asian_price_raw(samples.arithmetic_average, float(k))
    cv = asian_price_with_average_control(samples.arithmetic_average, float(k), params.f0)
    euro = european_price_estimate(samples.terminal, float(k))
    strike_rows.append({
        'strike': float(k),
        'asian_price_raw': raw['price'],
        'asian_stderr_raw': raw['stderr'],
        'asian_price_cv': cv['price'],
        'asian_stderr_cv': cv['stderr'],
        'european_price': euro['price'],
        'european_stderr': euro['stderr'],
        'asian_minus_european': cv['price'] - euro['price'],
        'cv_stderr_reduction': cv['stderr_reduction'],
    })

strike_df = pd.DataFrame(strike_rows)
strike_df


In [ ]:
fig, ax = plt.subplots()
ax.plot(strike_df['strike'], strike_df['european_price'], marker='o', label='European call')
ax.errorbar(
    strike_df['strike'],
    strike_df['asian_price_cv'],
    yerr=1.96 * strike_df['asian_stderr_cv'],
    marker='s',
    capsize=3,
    label='Arithmetic Asian call with CV',
)
ax.set_xlabel('Strike')
ax.set_ylabel('Monte Carlo price')
ax.set_title('Asian and European call prices across strikes')
ax.legend()
plt.show()


### 15.7 Summary Validation Table

The checks below are intentionally numerical rather than cosmetic. They test whether the path-dependent extension keeps the properties that matter for using the paper algorithm outside the European benchmark setting.


In [ ]:
asian_prices = strike_df['asian_price_cv'].to_numpy()
european_prices = strike_df['european_price'].to_numpy()
first_diff = np.diff(asian_prices)
second_diff = np.diff(asian_prices, n=2)

validation_rows = [
    {
        'check': 'single-step terminal samples match project simulator',
        'metric': float(single_step_check['max_abs_terminal_diff'].iloc[0]),
        'passed': bool(single_step_check['passed'].iloc[0]),
    },
    {
        'check': 'terminal samples are finite and nonnegative',
        'metric': float(np.min(samples.terminal)),
        'passed': bool(np.isfinite(samples.terminal).all() and np.all(samples.terminal >= 0.0)),
    },
    {
        'check': 'arithmetic averages are finite and nonnegative',
        'metric': float(np.min(samples.arithmetic_average)),
        'passed': bool(np.isfinite(samples.arithmetic_average).all() and np.all(samples.arithmetic_average >= 0.0)),
    },
    {
        'check': 'terminal martingale z-score within 3 standard errors',
        'metric': float(abs(terminal_diag['z_score'])),
        'passed': bool(abs(terminal_diag['z_score']) < 3.0),
    },
    {
        'check': 'average martingale z-score within 3 standard errors',
        'metric': float(abs(average_diag['z_score'])),
        'passed': bool(abs(average_diag['z_score']) < 3.0),
    },
    {
        'check': 'Asian prices decrease with strike',
        'metric': float(np.max(first_diff)),
        'passed': bool(np.all(first_diff <= 1e-12)),
    },
    {
        'check': 'Asian prices are convex in strike on the grid',
        'metric': float(np.min(second_diff)),
        'passed': bool(np.all(second_diff >= -1e-12)),
    },
    {
        'check': 'Asian prices are below same-path European prices',
        'metric': float(np.max(strike_df['asian_minus_european'])),
        'passed': bool(np.all(strike_df['asian_minus_european'] <= 3.0 * strike_df['asian_stderr_cv'])),
    },
    {
        'check': 'average control variate reduces Asian baseline stderr',
        'metric': float(asian_cv['stderr_reduction']),
        'passed': bool(asian_cv['stderr_reduction'] > 1.0),
    },
]

validation_summary = pd.DataFrame(validation_rows)
validation_summary


### 15.8 Optional Heavier Configuration

The default notebook is sized to run quickly. For a paper-scale appendix, increase the path budgets below and rerun the monitoring convergence section. The code is disabled by default to keep this notebook practical during review.


In [ ]:
RUN_HEAVY = False

if RUN_HEAVY:
    heavy_reference_paths = 300_000
    heavy_grid_paths = 200_000
    heavy_reference_contract = AsianContract(
        maturity=contract.maturity,
        strike=contract.strike,
        n_monitoring=504,
        include_initial=contract.include_initial,
    )
    print('Suggested heavy setup:')
    print('reference_paths =', heavy_reference_paths)
    print('grid_paths =', heavy_grid_paths)
    print('reference_monitoring =', heavy_reference_contract.n_monitoring)
else:
    print('Heavy run skipped. Set RUN_HEAVY = True to print the paper-scale settings.')


### 15.9 Asian Extension Conclusion

This extension is useful because it tests the SABR simulation scheme in the setting where Monte Carlo is actually needed: a path-dependent payoff with many monitoring dates.

The numerical evidence above supports four practical conclusions:

1. the Asian path simulator is consistent with the existing terminal SABR simulator in the one-step case;
2. the simulated terminal forward and arithmetic average remain close to their martingale expectation $F_0$;
3. the average-control-variate estimator materially reduces standard error without changing the model;
4. monitoring-frequency convergence is visible and can be measured against a fine-grid Monte Carlo proxy.

The main remaining limitation is benchmark availability. Unlike European calls, arithmetic Asian SABR prices do not have a simple FDM table in the paper, so the fine-grid Monte Carlo reference should be treated as an internal numerical benchmark rather than an external truth.


## 16. Optional CSV Export Helpers

Use these after running the sections you care about.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
# Example exports. Uncomment what you need.
# table1_df.to_csv(OUTPUT_DIR / 'table1.csv', index=False)
# table2_df.to_csv(OUTPUT_DIR / 'table2.csv', index=False)
# table4_df.to_csv(OUTPUT_DIR / 'table4.csv', index=False)
# table5_df.to_csv(OUTPUT_DIR / 'table5.csv', index=False)
# table6_df.to_csv(OUTPUT_DIR / 'table6.csv', index=False)
# table7_df.to_csv(OUTPUT_DIR / 'table7.csv', index=False)
# figure1_df.to_csv(OUTPUT_DIR / 'figure1_dataset.csv', index=False)
# figure3_df.to_csv(OUTPUT_DIR / 'figure3_dataset.csv', index=False)
